# Day 8 — Random Forest Baseline Training and Evaluation

## Objective
Train and evaluate a Random Forest classifier for telecom network anomaly detection using the public TelecomTS dataset.

## Research Question
Can Random Forest improve anomaly detection compared with the Logistic Regression baseline?

## Dataset
TelecomTS — Public Cellular Network KPI Dataset (Hugging Face)

## Feature Engineering
Sixteen numerical KPIs, each summarized using mean, standard deviation, minimum, and maximum, producing 64 features per sample.

## Model
Random Forest Classifier
- Number of trees: 100 (`n_estimators=100`)
- Random state: 42 (`random_state=42`)
- Parallel processing: Enabled (`n_jobs=-1`)

## Evaluation
Precision, recall, F1-score, ROC-AUC, PR-AUC, confusion matrix, and feature importance.

## Scope & Research Disclaimer
This is an independent research prototype using public data. The results do not represent measured performance or real-world fault data on SLTMobitel's network.

In [1]:
# Setup cell for Google Colab environments
# Installs Hugging Face datasets library silently if not already present
!pip install -q datasets

## Step 2 — Load TelecomTS Dataset

We load the public **TelecomTS** dataset directly from Hugging Face (`AliMaatouk/TelecomTS`).
This ensures the notebook runs independently from start to finish without depending on saved CSV files from earlier days.

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "AliMaatouk/TelecomTS",
    data_files={"full": "**/chunked.jsonl"}
)

data = dataset["full"]
print("Number of samples:", len(data))
print("Dataset columns:", data.column_names)

Number of samples: 32000
Dataset columns: ['start_time', 'end_time', 'sampling_rate', 'KPIs', 'description', 'anomalies', 'statistics', 'labels', 'QnA']


## Step 3 — Check the KPI Structure

Each record in TelecomTS contains time-series measurements for cellular radio and traffic KPIs, along with an anomaly label.
Let us inspect the KPI names and the structure of the anomaly label.

In [3]:
print("Available KPI names:")
print(list(data[0]["KPIs"].keys()))
print("\nExample anomaly label:")
print(data[0]["labels"]["anomaly_present"])

Available KPI names:
['RSRP', 'DL_BLER', 'DL_MCS', 'UL_BLER', 'UL_MCS', 'UL_NPRB', 'UL_SNR', 'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer', 'PRBs_DL_Current', 'PRBs_UL_Current', 'PRB_Utilization_DL', 'PRB_Utilization_UL', 'UL_Protocol', 'UL_NumberOfPackets', 'DL_Protocol', 'DL_NumberOfPackets']

Example anomaly label:
Yes


## Step 4 — Recreate the 64 Engineered Features

To represent the temporal KPI signals in tabular format for standard machine learning classifiers, we summarize each of the **16 numerical KPIs** using **4 statistical metrics** across its observation window:

1. **Mean**: Average signal level.
2. **Standard Deviation (std)**: KPI variability / stability.
3. **Minimum (min)**: Lowest observed value in the window.
4. **Maximum (max)**: Highest peak value in the window.

$$\text{Total Features} = 16 \text{ KPIs} \times 4 \text{ statistics} = 64 \text{ features}$$

Target encoding: **0 = Normal**, **1 = Anomaly**.

In [4]:
import pandas as pd

numerical_kpis = [
    "RSRP", "DL_BLER", "DL_MCS", "UL_BLER", "UL_MCS",
    "UL_NPRB", "UL_SNR", "TX_Bytes", "RX_Bytes", "Estimated_UL_Buffer",
    "PRBs_DL_Current", "PRBs_UL_Current", "PRB_Utilization_DL",
    "PRB_Utilization_UL", "UL_NumberOfPackets", "DL_NumberOfPackets"
]

feature_rows = []
for sample in data:
    row = {}
    for kpi in numerical_kpis:
        values = sample["KPIs"][kpi]
        row[f"{kpi}_mean"] = sum(values) / len(values)
        row[f"{kpi}_std"] = pd.Series(values).std()
        row[f"{kpi}_min"] = min(values)
        row[f"{kpi}_max"] = max(values)
    feature_rows.append(row)

X = pd.DataFrame(feature_rows)
y = pd.Series([
    1 if sample["labels"]["anomaly_present"] == "Yes" else 0
    for sample in data
])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts().sort_index())
print("\nTarget percentage:")
print((y.value_counts(normalize=True) * 100).sort_index())

Feature shape: (32000, 64)
Target shape: (32000,)

Target distribution:
0    30765
1     1235
Name: count, dtype: int64

Target percentage:
0    96.140625
1     3.859375
Name: proportion, dtype: float64


## Step 5 — Prepare Stratified Train-Test Split (80/20)

To ensure a strictly fair, scientific comparison with our Day 5 Logistic Regression baseline, we use the identical train-test split configuration:
- **Test size**: 20% (6,400 samples)
- **Training size**: 80% (25,600 samples)
- **Stratification**: `stratify=y` preserves the 3.86% anomaly prevalence across splits.
- **Random State**: `random_state=42` ensures deterministic sample allocation.

> **Important Viva Point (Tree Invariance to Feature Scaling):**  
> While Logistic Regression strictly required `StandardScaler` (because gradient updates and weight penalization depend on variable magnitude), **Random Forest does NOT require feature scaling**. Decision tree split points ($x_j \le \theta$) are scale-invariant and depend solely on the relative ordering of feature values.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining anomaly rate:", round(y_train.mean(), 4))
print("Testing anomaly rate:", round(y_test.mean(), 4))
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())
print("\nTesting class counts:")
print(y_test.value_counts().sort_index())

Training samples: 25600
Testing samples: 6400

Training anomaly rate: 0.0386
Testing anomaly rate: 0.0386

Training class counts:
0    24612
1      988
Name: count, dtype: int64

Testing class counts:
0    6153
1     247
Name: count, dtype: int64


## Step 6 — Train Random Forest Baseline Model

### Why Random Forest?
1. **Ensemble Learning (Bagging)**: Random Forest constructs $B=100$ independent decision trees using bootstrap samples (sampling with replacement) of the training dataset.
2. **Feature Subsampling ($max\_features=\sqrt{d}$)**: At each candidate split, the tree inspects a random subset of features ($\approx \sqrt{64} = 8$ features). This decorrelates the trees, preventing a few dominant KPIs from biasing every single tree.
3. **Nonlinear Partitioning**: Unlike Logistic Regression's flat hyperplane, Random Forest carves out orthogonal decision boundaries that capture complex, multi-KPI interactions (such as simultaneous SNR drops and BLER spikes).

We train the model using `n_estimators=100`, `random_state=42`, and `n_jobs=-1` (all CPU cores).

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("Random Forest training completed.")

Random Forest training completed.


## Step 7 — Evaluate the Random Forest Model

We now evaluate the trained ensemble on the unseen 6,400 testing samples.
We compute:
- Precision, Recall, and F1-score (for both Normal and Anomaly classes)
- Full Confusion Matrix (True Negatives, False Positives, False Negatives, True Positives)
- ROC-AUC (Area Under the Receiver Operating Characteristic Curve)
- PR-AUC / Average Precision (Area Under the Precision-Recall Curve)

In [7]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(
    y_test, y_pred, target_names=["Normal", "Anomaly"], digits=4
))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_test, y_prob), 4))

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9972    1.0000    0.9986      6153
     Anomaly     1.0000    0.9312    0.9644       247

    accuracy                         0.9973      6400
   macro avg     0.9986    0.9656    0.9815      6400
weighted avg     0.9974    0.9973    0.9973      6400

Confusion Matrix:
[[6153    0]
 [  17  230]]
True Negatives (TN): 6153
False Positives (FP): 0
False Negatives (FN): 17
True Positives (TP): 230
ROC-AUC: 0.9998
PR-AUC (Average Precision): 0.9954


### Empirical Model Comparison: Logistic Regression vs. Random Forest

| Performance Metric | Day 5 Logistic Regression Baseline | Day 8 Random Forest Baseline | Performance Impact / Delta | Interpretation for Telecom Operations |
| :--- | :--- | :--- | :--- | :--- |
| **Precision (Anomaly)** | 95.31% | **100.00%** | **+4.69%** | **Zero False Alarms**: When RF flags an incident, it is guaranteed genuine in this test set. |
| **Recall (Anomaly)** | 74.09% | **93.12%** | **+19.03%** | **Substantial Fault Coverage**: 230 out of 247 network anomalies successfully caught. |
| **F1-Score (Anomaly)** | 83.37% | **96.44%** | **+13.07%** | Major boost in balanced harmonic performance under 3.86% imbalance. |
| **ROC-AUC** | 97.85% | **99.98%** | **+2.13%** | Near-perfect class separation across all operating thresholds. |
| **PR-AUC (Avg Precision)**| ~88.42% | **99.54%** | **+11.12%** | Exceptionally high precision sustained across nearly the full recall curve. |
| **True Positives (TP)** | 183 | **230** | **+47** | Successfully detected 47 additional anomalous periods. |
| **False Positives (FP)** | 9 | **0** | **-9** | **0 false alarms** eliminates engineering alert fatigue completely. |
| **False Negatives (FN)** | 64 | **17** | **-47** | Missed anomalous incidents dropped from 64 down to 17 (73.4% reduction). |

### Key Takeaways for Viva:
1. **Nonlinear Advantage**: The dramatic jump in recall from 74.09% to 93.12% proves that network anomaly signatures are non-linearly coupled. Decision trees capture threshold boundaries that linear hyperplanes cannot.
2. **Zero False Positives**: While Day 6's class-weighting achieved 93.12% recall at the cost of **260 false alarms** (dropping precision to 46.94%), Random Forest achieves the **exact same 93.12% recall with ZERO false alarms** (100.00% precision)!

## Step 8 — Examine Feature Importance

Random Forest computes **Gini Importance** (also known as Mean Decrease in Impurity, MDI).
For each feature, MDI calculates the total decrease in node impurity (weighted by the probability of reaching that node) averaged over all 100 trees in the forest.

> **Cautionary Domain Note:**  
> High feature importance indicates that the model relies heavily on that KPI statistic to differentiate normal from abnormal states. It does **not** prove physical causation in a live radio access network.

In [8]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(15)

,Feature,Importance
0,RSRP_mean,0.112628
24,UL_SNR_mean,0.108233
3,RSRP_max,0.073713
12,UL_BLER_mean,0.067714
2,RSRP_min,0.051122
51,PRB_Utilization_DL_max,0.040632
32,RX_Bytes_mean,0.037634
48,PRB_Utilization_DL_mean,0.032807
22,UL_NPRB_min,0.027175
27,UL_SNR_max,0.026765


In [9]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(15).sort_values(by="Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"], color="#1f77b4")
plt.xlabel("Feature Importance (Mean Decrease in Impurity)")
plt.ylabel("KPI Feature")
plt.title("Top 15 Random Forest Features for Telecom Anomaly Detection")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Day 8 Conclusions & Summary

1. **Definitive Research Finding**: Moving from a linear model (Logistic Regression) to a non-linear ensemble (Random Forest) yielded substantial gains across all critical evaluation metrics.
2. **Elimination of False Alarms**: False positives were completely reduced from 9 down to 0, yielding a perfect **100.00% precision** on the test set.
3. **High Anomaly Recall**: Anomaly recall jumped from 74.09% to **93.12%**, reducing missed anomalies from 64 to just 17.
4. **Dominant KPIs**: Radio layer signal degradation (`RSRP_mean`, `UL_SNR_mean`), transmission error rates (`UL_BLER_mean`), and downlink resource congestion (`PRB_Utilization_DL_max`) are the most influential indicators of network anomalies.
5. **Viva Defense Readiness**: The experimental methodology adhered strictly to reproducible standards (`random_state=42`, stratified sampling, identical 64-feature representation, independent loading) while properly delineating public benchmark data from proprietary cellular network operations.